# pymefpp_partition_data

Notebook to create a partition of the calibration files for later reduction.

The user needs to indicate the following directories:

 - raw_data_dir: directory where the raw data is located
 - reduced_science_dir: directory root where the data reduction arena will be established.
 
This is the first notebook within the mefpp series, where each one is numbered from 01 to 07, indicating the order in which they must be run.

# Testing data

Notebook 1 run without any troubles.

In [ ]:
# importing needed system modules
import glob
import sys
import os
from pathlib import Path
import logging

# here we import the local mefpp_run library
# where the python scripts are
sys.path.insert(0, os.getcwd())
from mefpp_associate_reduction import *

# and define some paths
#root_dir = "/Users/fselman/Pictures/OCAM/CPPup/reduced/"  # the root directory where the calibrated and science images will go
#raw_data_dir = "/Users/fselman/Pictures/OCAM/CPPup/raw/" # where the raw images are
raw_data_dir = "/Users/fselman/Pictures/OCAM/M67/raw/data_with_raw_calibs/"
root_dir = "/Users/fselman/Pictures/OCAM/M67/"
reduced_cal_dir = root_dir+'calibs/'
reduced_science_dir = root_dir
log_dir = root_dir+'logs/'

# create a list with all the images in the raw data dir
listing = glob.glob(raw_data_dir + "OMEGA.*.fits")

In [ ]:
# create the root dir if not exist, then all the rest of folders
try:
    os.makedirs(reduced_cal_dir)
except Exception as e:
    print(e)
    pass

try:
    os.makedirs(reduced_science_dir)
    #os.makedirs(reduced_science_dir+'/logs/')
except Exception as e:
    print(e)
    pass
#create the log_dir outside the science dir.
try:
    os.makedirs(log_dir)
except Exception as e:
    print(e)
    pass

# we prepare the logs
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_dir+"/pymeffpp_partition_data.log"),
                              logging.StreamHandler()
                    ])
    

In [ ]:
# we find all OMEGACAM fits files below the raw data directory
logging.info('%d OmegaCAM frames in %s'%(len(listing), raw_data_dir))

exposurePartition = PartitionsTree()


i_exposure = 0

for file_name in listing:

    i_exposure+= 1
    exposure = Exposure(file_name)
    exposurePartition.insertExposure(exposure)
    biasDict = {k:v for (k,v) in exposurePartition.partList.items() if 'bias/' in k}
    domeDict = {k:v for (k,v) in exposurePartition.partList.items() if 'dome/' in k}
    skyDict = {k:v for (k,v) in exposurePartition.partList.items() if 'sky/' in k}
    stdDict = {k:v for (k,v) in exposurePartition.partList.items() if 'std/' in k}
    sciDict = {k:v for (k,v) in exposurePartition.partList.items() if 'science/' in k}
    if i_exposure%10:
        Total = len(biasDict) + len(domeDict) + \
                len(skyDict) + len(stdDict) +\
                len(sciDict)
        print("BIAS:     %d" % len(biasDict), \
                          "DOME:     %d" % len(domeDict), \
                          "SKY:      %d" % len(skyDict), \
                          "STD:      %d" % len(stdDict), \
                          "SCIENCE:  %d" % len(sciDict), \
                          "Total     %d" % Total, end='\r')


In [ ]:
# we create directory structure to reduce the calibration data

# bad pixel masks BPM are needed but static at this point we create
# a directory to contain them
try:
    os.makedirs(reduced_cal_dir+'bpm')
except:
    pass

Path(reduced_cal_dir+'bpm'+'/bpm.list').touch()
Path(reduced_cal_dir+'bpm'+'/toreduce_bpm').touch()


In [ ]:
# here we create a bias.list file in each bias partition
createFrameList(reduced_cal_dir, 'bias.list', biasDict)

In [ ]:
# here we create a dome.list file in dome partition
createFrameList(reduced_cal_dir, 'dome.list', domeDict)

In [ ]:
# here we create a sky.list file in each sky partition
createFrameList(reduced_cal_dir, 'sky.list', skyDict)

In [ ]:
# here we create a science.list file in each science partition
createFrameList(reduced_science_dir, 'science.list', sciDict)